# 01 — Classificatore a feature (TF-IDF + regressione logistica)

Sul dataset finale (`1-data-cleaning/03_dataset_finale`): solo il **testo degli articoli**.

Fold **per gruppo** (la valutazione principale: tutte le testate di un gruppo, cioè rete o stesso owner, nello
stesso fold) e **per testata** (confronto: i gruppi possono essere spezzati). Un modello per fold: vocabolario e
IDF si stimano **solo sul training** del fold, il test passa solo da `transform`. Iperparametri fissi.

Con `TAGLIO_TOKEN` il testo si taglia ai primi N token del tokenizer di ModernBERT: lo stesso testo che vede il
transformer con `MAX_TOKEN` = N.

Accanto al classificatore due baseline senza testo: la **classe di maggioranza** del training e una **scelta a
caso** con le proporzioni del training.

Si salva **tutto**, per poter ricalcolare qualsiasi metrica senza riaddestrare:

- per ogni articolo del test: `article_id`, `domain`, `fold`, `y_true`, i **logit** `logit_l1..logit_l5`, le
  **probabilità** `p_l1..p_l5`, `y_pred`;
- `metriche.json`: per articolo, per testata col voto e per testata con la media delle probabilità —
  accuratezza, F1 macro e pesata, precisione/recall/F1 per livello, matrice di confusione, tolerant 0.5 e 1,
  errore medio in livelli, binario (soglia $l_2$/$l_3$) con TP, TN, FP, FN, precisione, recall, F1.

File: `Results/2-classificazione/<NOME>_<schema>_articoli.csv`, `<NOME>_<schema>_metriche.json`,
`<NOME>_metriche.csv` (tabella riassuntiva), `<NOME>_FATTO` quando tutto è finito.

**Sul server:** `cd ~/John/WebScience2027/2-classificazione && papermill -k mbvenv 01_classificatore_feature.ipynb ../Results/2-classificazione/esecuzioni/01_classificatore_feature.ipynb`
(di solito lo lancia `coda_weekend.py`).

In [1]:
import json
import time

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

from comune import CLASSI, NOME_SCHEMA, SCHEMI, SEED, carica, cartella_risultati, letture, metriche, per_testata, registratore

In [2]:
NOME = "feature"
TAGLIO_TOKEN = 0           # 0 = testo intero; N = primi N token (tokenizer di ModernBERT)
SCHEMI_DA_FARE = SCHEMI
N_JOBS = 5                 # fold in parallelo
PROVA = False              # True: 40 testate, risultati in Results/2-classificazione/prova/

In [3]:
# Parameters
NOME = "feature"


In [4]:
OUT = cartella_risultati(PROVA)
registra = registratore(OUT / f"avanzamento_{NOME}.log")
art = carica(PROVA)
if TAGLIO_TOKEN:
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")
    enc = tok(art.text.tolist(), truncation=True, max_length=TAGLIO_TOKEN, add_special_tokens=False, return_offsets_mapping=True)
    art["text"] = [t[:o[-1][1]] if o else t for t, o in zip(art.text, enc["offset_mapping"])]
registra(f"=== {NOME}: {len(art)} articoli, {art.domain.nunique()} testate, taglio {TAGLIO_TOKEN or 'nessuno'} ===")

2026-10-18 09:43:33  === feature: 40001 articoli, 269 testate, taglio nessuno ===


## Addestramento

Per ogni fold: `fit_transform` sul training, `transform` sul test. I logit sono la `decision_function` della
regressione logistica.

In [5]:
def un_fold(schema, fold):
    t0 = time.time()
    tr, te = art[art[schema] != fold], art[art[schema] == fold]
    v = TfidfVectorizer(ngram_range=(1, 2), min_df=5, max_df=0.9, sublinear_tf=True,
                        max_features=100_000, strip_accents="unicode")
    clf = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)
    clf.fit(v.fit_transform(tr.text), tr.label)              # vocabolario, IDF e modello: solo training
    X = v.transform(te.text)                                   # test: solo transform
    registra(f"{schema} fold {fold}: fatto in {time.time() - t0:.0f} s ({len(tr)} training, {len(te)} test)")
    return te.index.to_numpy(), clf.classes_, clf.decision_function(X), clf.predict_proba(X)


def addestra(schema):
    file_out = OUT / f"{NOME}_{schema}_articoli.csv"
    if file_out.exists():
        registra(f"{schema}: già fatto, carico {file_out.name}")
        return pd.read_csv(file_out)
    logit, proba = np.full((len(art), 5), np.nan), np.zeros((len(art), 5))
    for idx, classi, z, p in Parallel(n_jobs=N_JOBS)(delayed(un_fold)(schema, f) for f in sorted(art[schema].unique())):
        logit[idx[:, None], classi[None, :]] = z
        proba[idx[:, None], classi[None, :]] = p
    pred = pd.DataFrame({"article_id": art.article_id, "domain": art.domain, "fold": art[schema], "y_true": art.label})
    for k in range(5):
        pred[f"logit_l{k + 1}"] = logit[:, k]
    for k, c in enumerate(CLASSI):
        pred[c] = proba[:, k]
    pred["y_pred"] = proba.argmax(1)
    pred.to_csv(file_out, index=False)
    json.dump(metriche(pred), open(OUT / f"{NOME}_{schema}_metriche.json", "w"), indent=1)
    return pred


pred = {s: addestra(s) for s in SCHEMI_DA_FARE}

2026-10-18 09:45:51  fold_gruppo fold 1: fatto in 136 s (30662 training, 9339 test)


2026-10-18 09:45:54  fold_gruppo fold 3: fatto in 137 s (31250 training, 8751 test)


2026-10-18 09:45:59  fold_gruppo fold 2: fatto in 143 s (31588 training, 8413 test)


2026-10-18 09:46:03  fold_gruppo fold 4: fatto in 146 s (33736 training, 6265 test)


2026-10-18 09:46:07  fold_gruppo fold 5: fatto in 149 s (32768 training, 7233 test)


2026-10-18 09:48:24  fold_testata fold 2: fatto in 135 s (31263 training, 8738 test)


2026-10-18 09:48:25  fold_testata fold 1: fatto in 136 s (30544 training, 9457 test)


2026-10-18 09:48:34  fold_testata fold 4: fatto in 143 s (32166 training, 7835 test)


2026-10-18 09:48:46  fold_testata fold 3: fatto in 155 s (32510 training, 7491 test)


2026-10-18 09:48:48  fold_testata fold 5: fatto in 156 s (33521 training, 6480 test)


## Baseline senza testo

Per ogni fold, dalle testate di training: la classe più frequente, e una classe estratta a caso con le
proporzioni del training (una per testata, seed fisso).

In [6]:
def baseline(schema, modo):
    rng = np.random.default_rng(SEED)
    y = np.zeros(len(art), dtype=int)
    for fold in sorted(art[schema].unique()):
        tr, te = art[art[schema] != fold], art[art[schema] == fold]
        priori = tr.drop_duplicates("domain").label.value_counts(normalize=True).sort_index()
        if modo == "maggioranza":
            y[te.index] = priori.idxmax()
        else:
            scelte = dict(zip(te.domain.unique(), rng.choice(priori.index, te.domain.nunique(), p=priori.values)))
            y[te.index] = te.domain.map(scelte)
    return pd.DataFrame({"domain": art.domain, "y_true": art.label, "y_pred": y})


for s in SCHEMI_DA_FARE:
    for modo in ["maggioranza", "a caso"]:
        json.dump(metriche(baseline(s, modo)), open(OUT / f"baseline_{modo.replace(' ', '_')}_{s}_metriche.json", "w"), indent=1)

## Risultati

Per articolo e per testata (voto di maggioranza): accuratezza a 5 classi, F1 macro, binario e tolerant.

In [7]:
righe = []
for s in SCHEMI_DA_FARE:
    righe.append({"modello": f"TF-IDF + LR ({NOME})", "fold": NOME_SCHEMA[s], **letture(pred[s])})
    for modo in ["maggioranza", "a caso"]:
        righe.append({"modello": f"baseline: {modo}", "fold": NOME_SCHEMA[s], **letture(baseline(s, modo))})
ris = pd.DataFrame(righe).set_index(["fold", "modello"])
ris.to_csv(OUT / f"{NOME}_metriche.csv")
ris[[c for c in ris.columns if c.startswith("testata")] + ["articolo 5cl", "articolo bin"]]

testata 5cl  testata F1  testata bin  \
fold        modello                                                       
per gruppo  TF-IDF + LR (feature)        0.357       0.282        0.677   
            baseline: maggioranza        0.353       0.104        0.643   
            baseline: a caso             0.212       0.176        0.528   
per testata TF-IDF + LR (feature)        0.472       0.414        0.747   
            baseline: maggioranza        0.353       0.104        0.643   
            baseline: a caso             0.223       0.196        0.543   

                                   testata tol0.5  testata tol1  articolo 5cl  \
fold        modello                                                             
per gruppo  TF-IDF + LR (feature)           0.556         0.755         0.286   
            baseline: maggioranza           0.498         0.643         0.351   
            baseline: a caso                0.383         0.554         0.207   
per testata TF-IDF + LR (feature)           0.639         0.807         0.380   
            baseline: maggioranza           0.498         0.643         0.351   
            baseline: a caso                0.394         0.565         0.203   

                                   articolo bin  
fold        modello                              
per gruppo  TF-IDF + LR (feature)         0.642  
            baseline: maggioranza         0.631  
            baseline: a caso              0.531  
per testata TF-IDF + LR (feature)         0.706  
            baseline: maggioranza         0.631  
            baseline: a caso              0.539

In [8]:
t = per_testata(pred[SCHEMI_DA_FARE[0]])
LIV = ["l1", "l2", "l3", "l4", "l5"]
display(pd.crosstab(t.y_true.map(dict(enumerate(LIV))), t.y_pred.map(dict(enumerate(LIV))), rownames=["vero"], colnames=["predetto"]))
(OUT / f"{NOME}_FATTO").write_text(time.strftime("%Y-%m-%d %H:%M:%S"))

predetto,l1,l2,l3,l4,l5
vero,,,,,
l1,7,14,2,8,2
l2,10,28,9,12,4
l3,4,15,5,10,3
l4,4,14,15,52,10
l5,1,12,0,24,4


19